# HyRoNA compilation pipeline

The source circuit is defined in `circuit.py`. This notebook selects the circuit and architecture, maps and schedules it with HyRoNA, and exports the visualization files.

## Build the source circuit

In [ ]:
from circuit import build_rotated_surface_code_circuit

source_circuit = build_rotated_surface_code_circuit(
    distance=3,
    max_ancillas=2,
    max_timeout=60,
)

print("Source operations:", source_circuit.count_ops())

In [ ]:
source_circuit.draw(output="mpl", fold=-1)

## Define the neutral-atom architecture

In [ ]:
from tempfile import NamedTemporaryFile

from mqt.qmap.hybrid_mapper import NeutralAtomHybridArchitecture

architecture_json = '''{
  "name": "example arch",
  "properties": {
    "nRows": 9,
    "nColumns": 9,
    "nAods": 1,
    "nAodCoordinates": 3,
    "interQubitDistance": 10,
    "minimalAodDistance": 1,
    "interactionRadius": 1,
    "blockingFactor": 2
  },
  "parameters": {
    "nQubits": 25,
    "gateTimes": {"none": 0.5, "x": 0.5},
    "gateAverageFidelities": {"none": 0.999, "x": 0.999},
    "decoherenceTimes": {"t1": 100000000, "t2": 1500000},
    "shuttlingTimes": {
      "move": 6.0,
      "aod_move": 6.0,
      "aod_activate": 1,
      "aod_deactivate": 1
    },
    "shuttlingAverageFidelities": {
      "move": 1,
      "aod_move": 1,
      "aod_activate": 1,
      "aod_deactivate": 1
    }
  }
}'''

with NamedTemporaryFile(mode="w", suffix=".json") as architecture_file:
    architecture_file.write(architecture_json)
    architecture_file.flush()
    architecture = NeutralAtomHybridArchitecture(architecture_file.name)

print(f"Loaded architecture: {architecture.name} ({architecture.num_qubits} qubits)")

## Map the circuit

In [ ]:
from mqt.core import load
from mqt.qmap.hybrid_mapper import (
    HybridNAMapper,
    InitialCircuitMapping,
    MapperParameters,
)

parameters = MapperParameters()
mapper = HybridNAMapper(architecture, params=parameters)
mqt_circuit = load(source_circuit)
mapper.map(mqt_circuit, initial_mapping=InitialCircuitMapping.graph)
mapping_stats = mapper.get_stats()
mapping_stats

## Schedule and export visualization files

In [ ]:
from pathlib import Path

schedule_stats = mapper.schedule(verbose=False, create_animation_csv=True)

output_dir = Path("visualisation_instructions")
output_dir.mkdir(parents=True, exist_ok=True)
output_base = output_dir / "hyrona"
mapper.save_animation_files(str(output_base))

print(f"Saved {output_base}.namachine and {output_base}.naviz")
schedule_stats

## Notes

HyRoNA accepts the circuit directly, so no separate native-basis transpilation is used. Change the builder in the first code cell to select another circuit from `circuit.py`. Mapper weights and the random seed can be adjusted through `MapperParameters`.